<a href="https://colab.research.google.com/github/hyunderscore/ds2002-fa26/blob/main/ds2002-fa26/tree/main/notebooks/03-pandas-cleaning/Copy%20of%202026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [48]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [49]:
# TODO
df['revenue'] = df['qty'] * df['price'] #create revenue by multiplying qty and price columns
print(df['revenue'].sum()) #sum revenue column
print(len(df)) #count number of rows

8520.0
400


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [50]:
# TODO
cat_rev = df.groupby('category')['revenue'].sum().reset_index() #create dataframe grouped by category with total revenue for each cat
cat_rev['percent'] = (cat_rev['revenue'] / df['revenue'].sum()) * 100 #add a column with the percent of total revenue
cat_rev = cat_rev.sort_values(by='revenue', ascending=False) #sort the dataframe by revenue
cat_rev

,category,revenue,percent
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [51]:
# TODO
avg_order = df.groupby('vendor_id')['revenue'].mean().reset_index() #create dataframe grouped by vendor with average revenue for each vendor
avg_order['order_ct'] = df.groupby('vendor_id')['revenue'].count().reset_index()['revenue'] #add a column with the order count for each vendor
avg_order #highest average order revenue is V-01

,vendor_id,revenue,order_ct
0,V-01,22.595745,94
1,V-05,20.580645,93
2,V-10,20.314286,105
3,V-18,21.750000,108


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [83]:
# TODO
print(f'{cat_rev[cat_rev['category'] == 'Merch']['percent'].iloc[0]:.1f}') #print the percentage from the cat rev df using an f string to round

20.8


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [76]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

merged_df = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one') #left merge and validate many to one

#print initial vs merged rows and revenue
print(f'initial rows {len(df)}')
print(f'merged rows {len(merged_df)}')

print(f'initial revenue {df["revenue"].sum()}')
print(f'merged revenue {merged_df["revenue"].sum()}')

#print the missing vendor then replace it with unknown vendor in the vendor name
print(merged_df[merged_df['vendor_name'].isna()]['vendor_id'].unique())
merged_df['vendor_name'] = merged_df['vendor_name'].fillna('Unknown Vendor')


merged_df.head()

initial rows 400
merged rows 400
initial revenue 8520.0
merged revenue 8520.0
['V-18']


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown Vendor
2,V-18,Drink,3,4.5,13.5,Unknown Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown Vendor


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [77]:
# TODO
merged_df.pivot_table(index='vendor_name', columns='category', values='revenue', aggfunc='sum', margins=True, margins_name='Total')

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [82]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(cat_rev['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(merged_df) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a). HoosBurger should focus on improving their drinks as where they are the highest revenue earner in terms of food (1338.0), they are the lowest in drinks (171.0), meaning if they improved their drink game they could easily out perform the other vendors. Rotunda Tacos needs to step up their game, as they are behind all of the other vendors in almost every category. All vendors are overall performing similarly.
b). Q3. The order counts and revenue seem to check out in theory but in practice there could still be major differences in statistics that are not accounted for such as number of items with each order or price per item.